In [1]:
!pip install -U transformers bitsandbytes accelerate

Defaulting to user installation because normal site-packages is not writeable


In [10]:
!pip uninstall torch torchvision -y
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.9.0
Uninstalling torch-2.9.0:
  Successfully uninstalled torch-2.9.0
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126


You can safely remove it manually.


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached https://download.pytorch.org/whl/cu126/torchvision-0.24.0%2Bcu126-cp313-cp313-win_amd64.whl.metadata (6.1 kB)
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB 15.4 MB/s eta 0:02:48
   ---------------------------------------- 0.0/2.6 GB 15.0 MB/s eta 0:02:53
   ---------------------------------------- 0.0/2.6 GB 14.8 MB/s eta 0:02:55
   ---------------------------------------- 0.0/2.6 GB 14.4 MB/s eta 0:02:59
   ---------------------------------------- 0.0/2.6 GB 14.4 MB/s eta 0:02:59
   ---------------------------------------- 0.0/2.6 GB 14.5 MB/s eta 0:02:57
   ---------------------------------------- 0.0/2.6 GB 14.5 MB/s eta 0:02:58
   ---------------------------------------- 0.0/2.6 GB 14.4 MB/s eta 0:02:58
   ---------------------------------------- 0.0/2.6 G

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [14]:
model_id = 'unsloth/Llama-3.2-1B-Instruct'
base_model = AutoModelForCausalLM.from_pretrained(model_id).to("cuda")

# INT8 Config
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
) #allows you to laod the model in 8-bit precision

model_8bit = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config_8bit, low_cpu_mem_usage=True) #passing the quantization config to the model

# 4 Bit Config
bnb_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True, #loaduing in 4 bit
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_4bit = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config_4bit, low_cpu_mem_usage=True)

# Loading Tokenizer
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-1B-Instruct")

# Print model size
print(f"Base Model size: {base_model.get_memory_footprint():,} bytes\n")
print(f"INT8 Model size: {model_8bit.get_memory_footprint():,} bytes\n") #model reduces size by almost half
print(f"4Bit Model size: {model_4bit.get_memory_footprint():,} bytes") #model reduces size by almost 75%

C:\Users\Adit\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Adit\.cache\huggingface\hub\models--unsloth--Llama-3.2-1B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to 

Base Model size: 4,943,257,728 bytes

INT8 Model size: 1,498,550,400 bytes

4Bit Model size: 1,012,011,136 bytes


In [15]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,)

In [16]:
# Weights from the First layer
base_weights = base_model.model.layers[0].self_attn.q_proj.weight.data
print("Original weights:")
print(base_weights)
print("Shape: ", base_weights.shape, "\n")
print("-" * 50, "\n") #These weights are represented in float16 format

# Weights From the First Layer - 8bit
weights_8bit = model_8bit.model.layers[0].self_attn.q_proj.weight.data
print("INT8 weights:")
print(weights_8bit)
print("Shape: ", weights_8bit.shape, "\n") #These weights are represented in int8 format

print("-" * 50, "\n")

# Weights From the First Layer - 4bit
weights_4bit = model_4bit.model.layers[0].self_attn.q_proj.weight.data
print("4Bit weights:")
print(weights_4bit)
print("Shape: ", weights_4bit.shape, "\n") #These weights are represented unsigned 8 format

Original weights:
tensor([[-0.0179,  0.0066,  0.0247,  ..., -0.0087, -0.0117,  0.0201],
        [ 0.0122,  0.0593,  0.0552,  ..., -0.0332, -0.0154,  0.0108],
        [ 0.0178,  0.0155,  0.0344,  ..., -0.0386, -0.0386, -0.0276],
        ...,
        [ 0.0298,  0.0352,  0.0713,  ..., -0.0718, -0.0265, -0.0287],
        [ 0.0226, -0.0248,  0.0352,  ..., -0.0120, -0.0287, -0.0148],
        [-0.0258, -0.0537, -0.0131,  ...,  0.0542,  0.0096, -0.0028]],
       device='cuda:0')
Shape:  torch.Size([2048, 2048]) 

-------------------------------------------------- 

INT8 weights:
tensor([[-44,  16,  61,  ..., -22, -29,  50],
        [ 14,  70,  65,  ..., -39, -18,  13],
        [ 16,  14,  30,  ..., -34, -34, -24],
        ...,
        [ 17,  20,  40,  ..., -40, -15, -16],
        [ 32, -35,  50,  ..., -17, -41, -21],
        [-14, -30,  -7,  ...,  30,   5,  -2]], device='cuda:0',
       dtype=torch.int8)
Shape:  torch.Size([2048, 2048]) 

-------------------------------------------------- 

4B

In [17]:
# Function for Generating With the Models
def generate_response(model, message, tokenizer):

    # Format message
    messages = [{"role": "user", "content": message}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95
    )

    # Decode
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "The capital of France is..."

base_response = generate_response(base_model, prompt, tokenizer)
print("Base Model Response:\n")
print(base_response)
print("-" * 50)

int8_response = generate_response(model_8bit, prompt, tokenizer)
print("INT8 Model Response:\n")
print(int8_response)
print("-" * 50)

bit4_response = generate_response(model_4bit, prompt, tokenizer)
print("4Bit Model Response:\n")
print(bit4_response)

Base Model Response:

system

Cutting Knowledge Date: December 2023
Today Date: 21 Oct 2025

user

The capital of France is...assistant

The capital of France is Paris.
--------------------------------------------------
INT8 Model Response:

system

Cutting Knowledge Date: December 2023
Today Date: 21 Oct 2025

user

The capital of France is...assistant

The capital of France is Paris.
--------------------------------------------------
4Bit Model Response:

system

Cutting Knowledge Date: December 2023
Today Date: 21 Oct 2025

user

The capital of France is...assistant

Paris.


In [18]:
#Calculate perplexity
def calculate_perplexity(model, text):
    # Encode the text
    encodings = tokenizer(text, return_tensors='pt').to("cuda")

    # Define input_ids and target_ids
    input_ids = encodings.input_ids
    target_ids = input_ids.clone()

    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)

    # Loss calculation
    neg_log_likelihood = outputs.loss

    # Perplexity calculation
    ppl = torch.exp(neg_log_likelihood)

    return ppl

In [19]:
#lower perplexity indicates better performance/confidence and accuracy
#higher perplexity indicates poor performance/confidence
base_perplexity = calculate_perplexity(base_model, base_response[41:])
print(f"Base Model Perplexity:  {base_perplexity.item():.2f}", "\n")
int8_perplexity = calculate_perplexity(model_8bit, int8_response[41:])
print(f"INT8 Model Perplexity:  {int8_perplexity.item():.2f}", "\n")
bit4_perplexity = calculate_perplexity(model_4bit, bit4_response[41:])
print(f"4Bit Model Perplexity:  {bit4_perplexity.item():.2f}")

Base Model Perplexity:  88.00 

INT8 Model Perplexity:  53.01 

4Bit Model Perplexity:  66.82
